# Peirce 1880：有限逻辑实验室

这是为 *On the Algebra of Logic*, *American Journal of Mathematics* 3(1), pp. 15–57 编写的原创教学重构，**不是历史软件复原**。使用 Python 标准库，把类解释为有限集合，把二元关系解释为有限有序对集合。

读者：理解集合、布尔值与 Python 函数即可。完成后，你能区分空主词的四类命题、执行有限消元、解释四种关系运算、检查分类边界，并用反例阻止不合法的量词交换。

路线：1. 类与存在承诺 → 2. 展开与消元 → 3. 原文算例核验 → 4. 关系、四运算与分类 → 5. 相等模式 → 6. 反例与测试。

页码均为原刊印刷页码。此 PDF 带一页封面，PDF 页码 = 印刷页码 − 13。公式依据扫描图像核对，不依据丢失上划线的 OCR。详见 SOURCE_LEDGER.md。

执行：已用随附 execute_notebook.py 在全新的共享 Python 命名空间中按顺序执行全部代码；这不是 Jupyter kernel 执行。Notebook 也可在安装好的 Jupyter 中重新启动内核并运行全部单元。

In [1]:
from pathlib import Path
import sys
# 在本目录或包含此目录的发布包根目录中运行。
if not Path('peirce_finite.py').exists():
    root = Path('reproduction/peirce-1880')
    if not (root / 'peirce_finite.py').exists():
        raise RuntimeError('请在 reproduction/peirce-1880 目录打开并执行此 notebook。')
    sys.path.insert(0, str(root.resolve()))
from peirce_finite import *
print('标准库有限模型已导入。')

标准库有限模型已导入。


## 1. 空主词仍可满足全称命题（p. 23）
A：所有 S 都是 P；E：没有 S 是 P；I：存在某个 S 是 P；O：存在某个 S 不是 P。
A 与 O 互为矛盾，E 与 I 互为矛盾。A、E 不承诺 S 非空。下面四行是原图四个区域的有限实例；不是把原图当作普通三段论对当方阵。

In [2]:
P = frozenset({0})
examples = [(1, frozenset({0})), (2, frozenset({0, 1})),
            (3, frozenset({1})), (4, frozenset())]
for quadrant, S in examples:
    print('区域', quadrant, 'S=', sorted(S), categorical(S, P))

区域 1 S= [0] {'A': True, 'E': False, 'I': True, 'O': False}
区域 2 S= [0, 1] {'A': False, 'E': False, 'I': True, 'O': True}
区域 3 S= [1] {'A': False, 'E': True, 'I': False, 'O': True}
区域 4 S= [] {'A': True, 'E': True, 'I': False, 'O': False}


区域 4 同时使 A、E 为真，使 I、O 为假。因此 A→I 与 E→O 都不能无条件使用。

## 2. 类的布尔运算与展开（pp. 33–36）
采用 ∪、∩、相对于明确宇宙 U 的补集。现代点值形式的 p.36 式 (26) 是：
f(x) = (f(1)∧x)∨(f(0)∧¬x)。每个对象的类成员资格都是一个布尔值。
这里按 p.36 式 (26) 编码，并由完整三变量真值函数测试核对。

In [3]:
f = lambda e: e['x'] or (e['y'] and e['z'])
for e in assignments(('x', 'y', 'z')):
    assert f(e) == shannon(f, 'x', e)
print('x OR (y AND z) 的 8 个赋值全部通过展开检查。')
canonical = cnf_from_truth_function(('x', 'y', 'z'), f)
print('完整子句展开：', format_cnf(canonical))

x OR (y AND z) 的 8 个赋值全部通过展开检查。
完整子句展开： (x OR ~y OR z) AND (x OR y OR ~z) AND (x OR y OR z)


上面生成三个包含全部变量的子句，对应 p.38 的例子。原文称这种完整子句为 “prime factors”，不能无说明地等同现代“极小素蕴含式”：现代化简后的 (x∨y)∧(x∨z) 只有两项。

### 一项方法论警告：格不一定是布尔代数
p.33 将分配律说成可由 (2)、(4) 推出。若**仅有**并、交的上下界规则，五元格 M₃ 就是反模型。这不否认集合布尔模型中的分配律，也不是对整篇论证体系的形式化裁决。

In [4]:
left = m3_meet('a', m3_join('b', 'c'))
right = m3_join(m3_meet('a', 'b'), m3_meet('a', 'c'))
print('M3: a meet (b join c) =', left)
print('M3: (a meet b) join (a meet c) =', right)
assert left != right

M3: a meet (b join c) = a
M3: (a meet b) join (a meet c) = 0


## 3. 从 p.39 的第四过程到有限布尔消元
原文规则：a⊆b∪x 且 c⊆d∪¬x，可推出 a∩c⊆b∪d。
令 a⊆x、x⊆b，就得到 a⊆b。现代逐对象编码为 (¬a∨x)∧(¬x∨b)，删除 x 得 ¬a∨b。

程序对所有含 x 和 ¬x 的子句配对，去掉主元，删除重言式和被包含的冗余子句。返回的公式刻画 ∃x F 的布尔投影。它处理**全称类包含的逐对象约束**；I/O 的存在证人没有在这个消元器内表示。

In [5]:
premises = normalize([clause('~a', 'x'), clause('~x', 'b')])
result, trace = eliminate(premises, 'x')
print('前提：', format_cnf(premises))
for step in trace:
    print('消元记录：', step)
print('结果：', format_cnf(result))

前提： (~a OR x) AND (b OR ~x)
消元记录： {'positive': '~a OR x', 'negative': 'b OR ~x', 'resolvent': '~a OR b', 'discarded_tautology': False}
结果： (~a OR b)


## 4. 直接核对原文 Boole 算例（pp. 39–42）
三个原始前提（补集记 ¬）：
1. ¬x∩¬z ⊆ v∩[(y∩¬w)∪(¬y∩w)]
2. ¬v∩x∩w ⊆ (y∩z)∪(¬y∩¬z)
3. (x∩y)∪(v∩x∩¬y) = (z∩¬w)∪(¬z∩w)

先用真值函数生成完整子句，再消去 v。独立检查：对剩余每个赋值直接枚举 v=0、1，并与 p.41 的六个前提比较。

In [6]:
original = cnf_from_truth_function(('v', 'x', 'y', 'z', 'w'), boole_original)
projected, _ = eliminate(original, 'v')
allowed, extra = [], []
for e in assignments(('x', 'y', 'z', 'w')):
    direct = any(boole_original(dict(e, v=v)) for v in (False, True))
    assert direct == eval_cnf(projected, e) == eval_cnf(boole_six_clauses(), e) == boole_exact_solution(e)
    row = ''.join(str(int(e[k])) for k in ('x', 'y', 'z', 'w'))
    if direct:
        allowed.append(row)
    if boole_printed_summary(e) and not direct:
        extra.append(row)
print('合法 xyzw 赋值（8个）：', ', '.join(allowed))
print('p.41 六前提：', format_cnf(boole_six_clauses()))
print('p.42 最末式额外接受：', ', '.join(extra))

合法 xyzw 赋值（8个）： 0011, 0100, 0111, 1000, 1001, 1010, 1101, 1110
p.41 六前提： (~w OR x OR z) AND (w OR x OR ~z) AND (~w OR ~x OR ~y OR ~z) AND (~w OR ~x OR y OR ~z) AND (w OR ~x OR ~y OR z) AND (w OR x OR y OR z)
p.42 最末式额外接受： 1011, 1100, 1111


**原文与重构不能混写。** p.42 最后显示 ∞⊆x∪(z∩w)∪(y∩¬z∩¬w)，并称它总结四个变量的所有关系。按本笔记声明的布尔语义，它只是必要条件，不等价于 p.41 六前提。它允许 11 行，正确投影允许 8 行；x=y=z=w=1 就是简单反例。我们保留原式并标注差异，没有默默“修复”原文。

现代化简的完整约束是 x=F，其中 F=(z∩¬w)∪(¬z∩w)∪(¬y∩¬z∩¬w)。p.42 最后显示的式子只保留了 F⊆x，遗漏 x⊆F。

另一方面，p.41 所说“再消去 x 后，y,z,w 之间没有独立限制”，确实可在这里检查。

In [7]:
without_x, _ = eliminate(projected, 'x')
assert all(eval_cnf(without_x, e) for e in assignments(('y', 'z', 'w')))
print('消去 v、x 后：', format_cnf(without_x))

消去 v、x 后： TRUE


## 5. 从类到二元关系（pp. 43–46）
R 的每个元素是有序对 (x,y)。补集相对于 U×U；逆关系交换两个坐标。
本重构统一约定 R;S 的路径方向：先沿 R，从 x 到 y，再沿 S，从 y 到 z。历史文献、线性代数与编程库可能使用不同书写方向，不能只凭符号猜测。

In [8]:
R = Relation(3, frozenset({(0, 0), (0, 1), (1, 1)}))
S = Relation(3, frozenset({(0, 2), (1, 2), (2, 0)}))
print('R 的 0/1 矩阵：', R.matrix())
print('R;S：', sorted(R.compose(S).pairs))
print('(0,2) 的中间证人：', R.witnesses(S, 0, 2))
assert R.compose(S).converse() == S.converse().compose(R.converse())
print('逆关系反转复合次序：通过。')

R 的 0/1 矩阵： ((1, 1, 0), (0, 1, 0), (0, 0, 0))
R;S： [(0, 2), (1, 2)]
(0,2) 的中间证人： (0, 1)
逆关系反转复合次序：通过。


## 6. 四种运算是四种明确语义（pp. 50–53）
用 comp、reg、prog、trans 名称避免无法可靠复制的历史上下标：
- comp(R,S) = R;S：∃y [Rxy∧Syz]
- reg(R,S) = ¬(R;¬S)：∀y [Rxy→Syz]，regressive involution
- prog(R,S) = ¬(¬R;S)：∀y [Syz→Rxy]，progressive involution
- trans(R,S) = ¬R;¬S：∃y [¬Rxy∧¬Syz]，transaddition

p.52 三标记分别对应 (|||)、(|--)、(-|-)、(--|)。注意 trans 没有再在结果外取补集。

In [9]:
for name in ('compose', 'regressive', 'progressive', 'transadd'):
    print(name, sorted(getattr(R, name)(S).pairs))
print('空的 R 行使 reg 对该行的每个终点为真：', R.regressive(S).matrix()[2])
assert R.regressive(S).matrix()[2] == (1, 1, 1)

compose [(0, 2), (1, 2)]
regressive [(0, 2), (1, 2), (2, 0), (2, 1), (2, 2)]
progressive [(0, 1), (0, 2), (1, 1), (2, 1)]
transadd [(0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]
空的 R 行使 reg 对该行的每个终点为真： (1, 1, 1)


reg 对 R 反单调、对 S 单调；prog 对 R 单调、对 S 反单调；trans 对两个输入都反单调。这解释了 p.52 的四组包含规则。普通复合对两个输入都单调。

## 7. 分类必须纳入 p.57 的勘误
设 I 为对角线，D=(U×U)\I。concurrent 表示 R⊆I；opponent 表示 R 与 D 相交非空；self-relative 表示与 I 相交非空；alio-relative 表示不含对角元素。
“negative of concurrent” 表示 **¬R 是 concurrent**，不是 “R 不是 concurrent”。零关系同时是 concurrent 和 alio-relative；全关系同时是两者的 negative。

In [10]:
for label, relation in [('0', Relation.zero(2)), ('I', Relation.identity(2)),
                         ('D', ~Relation.identity(2)), ('top', Relation.top(2))]:
    print(label, [k for k, v in relation.classify().items() if v])
assert Relation.zero(2).classify()['concurrent']
assert Relation.zero(2).classify()['alio_relative']
assert Relation.top(2).classify()['negative_of_concurrent']
assert Relation.top(2).classify()['negative_of_alio_relative']

0 ['concurrent', 'alio_relative', 'negative_of_opponent', 'negative_of_self_relative']
I ['concurrent', 'self_relative', 'negative_of_opponent', 'negative_of_alio_relative']
D ['opponent', 'alio_relative', 'negative_of_concurrent', 'negative_of_self_relative']
top ['opponent', 'self_relative', 'negative_of_concurrent', 'negative_of_alio_relative']


## 8. 相等模式与 Bell 数（p. 48）
三元组有五种相等模式：AAA、AAB、ABA、ABB、ABC。用首次出现次序把对象改名为 0、1、2，得到受限增长串。代码通过这种构造枚举模式，又用独立的 Stirling 递推计算 Bell 数来核对。
原文三元形式 (B:A):A 对应 ABB 模式；这里只有相等结构，字母名本身不重要。

In [11]:
print('三元模式：', equality_patterns(3))
print('B0..B7：', bell_numbers(7))
print('原文列出的 B1..B6：', bell_numbers(6)[1:])
print('一个重新命名实例：', equality_pattern(('oak', 'pine', 'oak')))
assert tuple(len(equality_patterns(n)) for n in range(8)) == bell_numbers(7)

三元模式： ((0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1), (0, 1, 2))
B0..B7： (1, 1, 2, 5, 15, 52, 203, 877)
原文列出的 B1..B6： (1, 2, 5, 15, 52, 203)
一个重新命名实例： (0, 1, 0)


有限宇宙大小也重要：若只有两个对象，ABC 无法实现。因此 Bell 数表示可用对象足够多时的相等模式总数；固定大小 n 的宇宙只能实现块数不超过 n 的模式。

## 9. 两个证人不等于同一个证人（p.55 的适用边界）
普通关系复合对并集分配，却一般不对交集分配。下面 R;S 与 R;T 分别能到达 (0,2)，但中间点分别是 0 和 1。不能把两个存在量词合并成一个共同证人。

In [12]:
r = Relation(3, frozenset({(0, 0), (0, 1)}))
s = Relation(3, frozenset({(0, 2)}))
t = Relation(3, frozenset({(1, 2)}))
print('分别的中间证人：', r.witnesses(s, 0, 2), r.witnesses(t, 0, 2))
print('r;(s∩t)：', sorted(r.compose(s & t).pairs))
print('(r;s)∩(r;t)：', sorted((r.compose(s) & r.compose(t)).pairs))
assert r.compose(s & t) != r.compose(s) & r.compose(t)

分别的中间证人： (0,) (1,)
r;(s∩t)： []
(r;s)∩(r;t)： [(0, 2)]


## 10. 量词交换与混合结合律（pp. 56–57）
普通复合的结合律可由两个存在量词交换得到。含全称量词的其他运算不能一概照搬。这里 a=top、b=I、c=top：左式要求先找到一个对所有对象都适用的共同中间点；右式可针对不同对象选不同证人。

In [13]:
a, b, c = Relation.top(2), Relation.identity(2), Relation.top(2)
left, right = a.regressive(b).compose(c), a.regressive(b.compose(c))
print('(a reg b);c：', sorted(left.pairs))
print('a reg (b;c)：', sorted(right.pairs))
assert left == Relation.zero(2) and right == Relation.top(2)
each_has_one = all(any((x, y) in b.pairs for y in range(2)) for x in range(2))
one_for_all = any(all((x, y) in b.pairs for x in range(2)) for y in range(2))
print('∀x∃y Ixy =', each_has_one, '; ∃y∀x Ixy =', one_for_all)

(a reg b);c： []
a reg (b;c)： [(0, 0), (0, 1), (1, 0), (1, 1)]
∀x∃y Ixy = True ; ∃y∀x Ixy = False


### 练习
1. 将上面的宇宙改成一个对象，量词交换反例还成立吗？为什么？
2. 在原始消元示例再加一个前提 b⊆c，消去 x、b，预期得到什么？

下一单元提供可运行答案。

In [14]:
# 答案 1：单元素宇宙的两个量词只有同一个候选值，反例消失。
i = Relation.identity(1)
assert all(any((x, y) in i.pairs for y in range(1)) for x in range(1))
assert any(all((x, y) in i.pairs for x in range(1)) for y in range(1))
# 答案 2：a⊆c。
f = normalize([clause('~a', 'x'), clause('~x', 'b'), clause('~b', 'c')])
f, _ = eliminate(f, 'x')
f, _ = eliminate(f, 'b')
print('消去 x、b：', format_cnf(f))
assert f == normalize([clause('~a', 'c')])

消去 x、b： (~a OR c)


## 11. 真正运行测试，而非只展示代码
随附测试由 unittest 自动发现，包含对所有指定小域模型的枚举及独立语义 oracle。每个穷举范围都有精确计数断言；范围见 README。下面执行整个测试套件。

In [15]:
import subprocess
import peirce_finite
root = Path(peirce_finite.__file__).parent
run = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-v'],
                     cwd=root, capture_output=True, text=True)
print((run.stdout + run.stderr).strip())
assert run.returncode == 0

test_boolean_laws_all_class_triples_n0_to_3 (tests.test_peirce_finite.ClassLogicTests.test_boolean_laws_all_class_triples_n0_to_3) ... ok
test_categorical_all_subject_predicate_pairs_n0_to_3 (tests.test_peirce_finite.ClassLogicTests.test_categorical_all_subject_predicate_pairs_n0_to_3) ... ok
test_expansion_all_256_ternary_boolean_functions (tests.test_peirce_finite.ClassLogicTests.test_expansion_all_256_ternary_boolean_functions) ... ok
test_m3_is_a_lattice_but_not_distributive (tests.test_peirce_finite.ClassLogicTests.test_m3_is_a_lattice_but_not_distributive) ... ok
test_p39_elimination_rule_all_five_class_assignments_n0_to_2 (tests.test_peirce_finite.ClassLogicTests.test_p39_elimination_rule_all_five_class_assignments_n0_to_2) ... ok
test_all_512_cnf_subsets_of_two_variable_canonical_clauses (tests.test_peirce_finite.EliminationTests.test_all_512_cnf_subsets_of_two_variable_canonical_clauses) ... ok
test_all_729_ordered_clause_pairs_on_three_variables (tests.test_peirce_finite.Elim

## 限制与延伸
- 有限穷举不等于所有宇宙上的一般证明。本程序不重建 pp.42–43 的连续性/理想极限，也不验证全部 pp.55–57 的发展公式。
- 消元器只处理布尔 CNF；不自动维护 I/O 的存在承诺，也不实现完整关系演算判定过程。
- 关系总数是 2^(n²)，全关系三元组数量是 2^(3n²)。n=3 就有 134,217,728 个三元组，因此这里的三元穷举只到 n=2；主测试在 n=3 做 512 个单关系检查及具体反例。另附 independent_checks.py 使用独立位掩码量词 oracle，额外穷举 n=3 的全部 262,144 个关系对子；仍没有穷举 n=3 三元组。
- 模型允许 n=0、1 来检测软件边界；这不表示 Peirce 的无限理想宇宙与这些退化宇宙相同。
- 代码是原创现代教学实现。现代 SAT、关系数据库和量词语义只能作为阅读桥梁，不应倒称为原文已发明其全部现代体系。

可选延伸：选择一条 p.55 的发展公式，先精确核对上划线与上下标，再写独立量词语义以及有边界的穷举测试；不要把整页 OCR 直接转换成代码。